# Notebook 5: RAG with a Vector Database

**Who this is for:** Digital Humanities students with no prior AI or programming background.

**What you will learn**
1. How a vector database does the "retrieve" step automatically
2. The three RAG steps in code: retrieve, build the prompt, generate
3. How to check the sources behind every answer
4. How to ask in Arabic and read English sources (and the other way round)
5. What happens when retrieval is too narrow

**Before you start:** Notebooks 2 (vector database) and 4 (RAG with static sources) explain the ideas used here. You need an API key (see Notebook 3, section 0). Lines that start with `#` are comments for humans.

## 0. Setup

In [ ]:
# Install the libraries for the vector database, the embeddings and the three providers.
!pip install -q chromadb sentence-transformers anthropic openai google-genai pandas matplotlib

In [ ]:
import os
from getpass import getpass

# Choose ONE provider. You only need a key for the one you choose (see Notebook 3, section 0).
PROVIDER = "claude"        # "claude", "openai" or "gemini"

# Model names are just settings. Change them if a name is retired (see each provider's documentation).
CLAUDE_MODEL = "claude-sonnet-4-5"
OPENAI_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"

# The name under which the key is saved in Colab Secrets (the key icon in the left sidebar).
KEY_NAMES = {"claude": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY", "gemini": "GEMINI_API_KEY"}

def get_key(name):
    """Read the key from Colab Secrets; if it is missing, ask for it (the typing is hidden)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name) or getpass(f"Enter {name}: ")

API_KEY = get_key(KEY_NAMES[PROVIDER])
print("Key found:", bool(API_KEY))      # prints True or False, never the key itself

In [ ]:
def ask(prompt, system=None, max_tokens=500):
    """Send a prompt to the chosen provider and return the answer as text."""

    if PROVIDER == "claude":
        import anthropic
        client = anthropic.Anthropic(api_key=API_KEY)
        options = dict(model=CLAUDE_MODEL, max_tokens=max_tokens, temperature=0,
                       messages=[{"role": "user", "content": prompt}])
        if system:
            options["system"] = system
        return client.messages.create(**options).content[0].text

    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=API_KEY)
        messages = [{"role": "system", "content": system}] if system else []
        messages.append({"role": "user", "content": prompt})
        reply = client.chat.completions.create(model=OPENAI_MODEL, messages=messages,
                                               temperature=0, max_tokens=max_tokens)
        return reply.choices[0].message.content

    if PROVIDER == "gemini":
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=API_KEY)
        config = types.GenerateContentConfig(system_instruction=system, temperature=0,
                                             max_output_tokens=max_tokens)
        return client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=config).text

    raise ValueError("PROVIDER must be 'claude', 'openai' or 'gemini'")

# Quick test: the answer should be the word "ready".
print(ask("Reply with one word: ready", max_tokens=10))

> Some newer models fix the temperature or ignore it. If you see an error mentioning `temperature`, delete `temperature=0` from the code above.

## 1. Build the database

We use the invented *Cedar Valley Reading Society* archive from Notebook 4 (12 records, each in English and Arabic). **Everything in it is invented**, so no model can know it already.

In [ ]:
#@title Load the invented Cedar Valley archive (double-click to show the code) { display-mode: "form" }
ARCHIVE = [
    {"id": "r01", "title": "Founding",
     "en": "The Cedar Valley Reading Society was founded in 1921 by the teacher Salim Haddad in a rented room above a bakery.",
     "ar": "تأسست جمعية وادي الأرز للقراءة عام 1921 على يد المعلم سليم حداد في غرفة مستأجرة فوق مخبز."},
    {"id": "r02", "title": "Library size",
     "en": "In 1930 the library of the society held 1,200 volumes, mostly Arabic poetry, history books and translated novels.",
     "ar": "في عام 1930 ضمّت مكتبة الجمعية 1200 مجلد، معظمها دواوين شعر عربي وكتب تاريخ وروايات مترجمة."},
    {"id": "r03", "title": "Members in 1928",
     "en": "The lending register for 1928 lists 340 members. Each member could borrow two books for fourteen days.",
     "ar": "يسجّل دفتر الإعارة لعام 1928 أسماء 340 عضوًا. وكان يحق لكل عضو استعارة كتابين لمدة أربعة عشر يومًا."},
    {"id": "r04", "title": "Late fines",
     "en": "A book returned late cost the borrower a fine of one piaster per day. The money was used to repair damaged books.",
     "ar": "كانت غرامة تأخير إعادة الكتاب قرشًا واحدًا عن كل يوم، وكان المال يُستخدم لإصلاح الكتب التالفة."},
    {"id": "r05", "title": "New building",
     "en": "In 1936 the society moved to a new building near the old market. The move was paid for by donations from local merchants.",
     "ar": "في عام 1936 انتقلت الجمعية إلى مبنى جديد قرب السوق القديم، ودُفعت تكاليف الانتقال من تبرعات تجار المدينة."},
    {"id": "r06", "title": "Monthly bulletin",
     "en": "From 1925 to 1939 the society published a monthly bulletin. Only 61 issues survive today, and they are kept in the municipal archive.",
     "ar": "من عام 1925 إلى عام 1939 أصدرت الجمعية نشرة شهرية. لم يبق منها اليوم سوى 61 عددًا، وهي محفوظة في أرشيف البلدية."},
    {"id": "r07", "title": "Women's reading circle",
     "en": "In 1931 the society started a weekly evening reading circle for women, led by the teacher Nadia Khoury.",
     "ar": "في عام 1931 بدأت الجمعية حلقة قراءة مسائية أسبوعية للنساء، أدارتها المعلمة نادية خوري."},
    {"id": "r08", "title": "Poetry lecture",
     "en": "In 1927 the visiting poet Khalil Saba gave a lecture on modern Arabic poetry, and about 200 people attended.",
     "ar": "في عام 1927 ألقى الشاعر الزائر خليل سابا محاضرة عن الشعر العربي الحديث، وحضرها نحو 200 شخص."},
    {"id": "r09", "title": "Merchant's donation",
     "en": "In 1933 the merchant Yusuf Kassab donated 300 books from his private library, including a rare copy of a seventeenth-century geography manuscript.",
     "ar": "في عام 1933 تبرّع التاجر يوسف كسّاب بـ 300 كتاب من مكتبته الخاصة، من بينها نسخة نادرة من مخطوط جغرافي يعود إلى القرن السابع عشر."},
    {"id": "r10", "title": "Closure",
     "en": "The society closed in 1951 when its funds ran out. Its books were kept in a school storeroom, and about 900 volumes were catalogued again in 1990.",
     "ar": "أُغلقت الجمعية عام 1951 بعد نفاد أموالها. حُفظت كتبها في مخزن مدرسة، وأُعيد فهرسة نحو 900 مجلد منها عام 1990."},
    {"id": "r11", "title": "Digitisation",
     "en": "In 2019 a university team digitised the surviving bulletins and the 1928 lending register. The scans can be read online for free.",
     "ar": "في عام 2019 رقمن فريق جامعي النشرات الباقية ودفتر الإعارة لعام 1928، ويمكن قراءة النسخ الممسوحة مجانًا على الإنترنت."},
    {"id": "r12", "title": "Reading habits",
     "en": "The 1928 register shows that poetry was the most borrowed category, followed by history and then translated novels.",
     "ar": "يُظهر سجل عام 1928 أن الشعر كان أكثر الأصناف استعارة، يليه التاريخ ثم الروايات المترجمة."},
]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import chromadb
from sentence_transformers import SentenceTransformer

# The multilingual embedding model from Notebook 1 (the first run downloads about 450 MB).
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Store every record twice: once in English and once in Arabic, each with its labels.
ids, texts, metas = [], [], []
for r in ARCHIVE:
    for lang in ["en", "ar"]:
        ids.append(f"{r['id']}_{lang}")
        texts.append(r[lang])
        metas.append({"record": r["id"], "title": r["title"], "language": lang})

db = chromadb.EphemeralClient()
try:
    db.delete_collection("archive")          # makes the cell safe to run twice
except Exception:
    pass
collection = db.create_collection(name="archive", metadata={"hnsw:space": "cosine"})
collection.add(ids=ids, embeddings=embedder.encode(texts).tolist(), documents=texts, metadatas=metas)
print("Stored", collection.count(), "texts")

## 2. The three RAG steps as functions

Each step from Notebook 4 becomes a small function.

In [ ]:
def retrieve(question, k=3, language=None):
    """Step 1 - RETRIEVE: find the k stored texts closest in meaning to the question.
    `language` can be "en" or "ar" to search only the English or only the Arabic texts."""
    options = dict(query_embeddings=[embedder.encode(question).tolist()], n_results=k)
    if language:
        options["where"] = {"language": language}
    found = collection.query(**options)
    return [{"score": round(1 - found["distances"][0][i], 2),       # similarity (1 = same meaning)
             "title": found["metadatas"][0][i]["title"],
             "text": found["documents"][0][i]}
            for i in range(len(found["ids"][0]))]

def build_prompt(question, passages):
    """Step 2 - AUGMENT: put the retrieved passages in the prompt as numbered sources."""
    sources = "".join(f"[{n}] {p['text']}\n" for n, p in enumerate(passages, start=1))
    return (
        "Answer the question using ONLY the sources below.\n"
        "Cite the numbers of the sources you used, for example [1].\n"
        "If the sources do not contain the answer, say exactly: The sources do not say.\n"
        "Answer in the same language as the question.\n\n"
        f"Sources:\n{sources}\n"
        f"Question: {question}"
    )

def rag(question, k=3, language=None):
    """Run all three steps, then print the answer AND the sources that were used."""
    passages = retrieve(question, k=k, language=language)     # 1. Retrieve
    answer = ask(build_prompt(question, passages))            # 2. Augment  3. Generate
    print("Question:", question)
    print("Answer  :", answer)
    print("\nSources used:")
    display(pd.DataFrame(passages)[["score", "title", "text"]])

## 3. Ask questions

Always read the **sources** table under the answer. This is how you check that the model did not invent anything.

In [ ]:
rag("Who founded the society, and where did it first meet?", language="en")

In [ ]:
# Arabic question, searching the Arabic texts. ("Which kind of books did members borrow most?")
rag("ما أكثر أنواع الكتب التي استعارها الأعضاء؟", language="ar")

In [ ]:
# Arabic question, searching the ENGLISH texts. The model reads English and answers in Arabic.
# ("Where can the surviving bulletins be found?")
rag("أين توجد النشرات الباقية؟", language="en")

## 4. A question the archive cannot answer

In [ ]:
# The archive says nothing about the society's budget.
rag("What was the yearly budget of the society?", language="en")

The retrieval step **always** returns the closest passages, even when none of them is relevant. The instruction in our prompt is what makes the model admit that the answer is missing. Look at the low `score` values: they are a warning sign.

## 5. Seeing the retrieval scores

The chart shows how close the six best passages are to a question. A clear winner means a confident search.

In [ ]:
question = "Who gave books to the society?"
top = retrieve(question, k=6, language="en")

plt.figure(figsize=(7, 3.5))
plt.barh([p["title"] for p in top][::-1], [p["score"] for p in top][::-1], color="#2a6f97")
plt.xlabel("similarity to the question (1 = same meaning)")
plt.title(f'Best passages for: "{question}"')
plt.tight_layout()
plt.show()

## 6. How many passages should we retrieve?

The number `k` matters. If it is too small, the answer may need a passage that was left out. The next question needs **two** records: the members and the most borrowed category.

In [ ]:
two_part = "How many members did the register list in 1928, and which category was borrowed most?"

print("=== k = 1 ===")
rag(two_part, k=1, language="en")

In [ ]:
print("=== k = 3 ===")
rag(two_part, k=3, language="en")

## Key takeaways

- RAG with a vector database: **retrieve** the closest passages, **put them in the prompt**, **generate** the answer.
- **Always read the sources** returned with the answer. This is your evidence.
- Retrieval always returns something. Low scores and the "sources do not say" rule protect you from invented answers.
- `k` (how many passages) is a setting to test, not a fixed rule.

**About long documents:** our records are short. Real documents (books, reports, letters) are first cut into **chunks** of a paragraph or two, and each chunk is stored in the database like our records.

## Exercises

1. Replace the archive with 5 to 10 paragraphs from your own research and ask three questions.
2. Compare `language="en"`, `language="ar"` and no language filter for the same Arabic question.
3. Try `k=2` and `k=5` on a question of your own. When does a larger `k` stop helping?

**Next:** Notebook 6 shows how to run an open-weight model yourself. Notebook 7 repeats this RAG notebook with that model instead of an API.